In [62]:
import requests
import pandas as pd
import numpy as np
import json

from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

In [2]:
api_key = 'd0df995498ccdad53117ca6a3b45f0ab'

headers = {
    'accept': 'application/json',
    'apikey': f'{api_key}'
}

In [70]:
features = [
    # community
    'geographyName',

    'median_Household_Income',
    'household_Income_Per_Capita',
    'housing_Median_Rent',
    'housing_Owner_Households_Median_Value',
    'housing_Units_Renter_Occupied_Pct',
    'housing_Units_Vacant_Pct',
    'median_Length_Of_Residence_Yr',
    'median_Age',
    'households_1_Person_Pct',
    'crime_Index',
    'annual_Avg_Temp',
    'possible_Sunshine_Pct',
    'air_Pollution_Index',

    'housing_Median_Built_Yr',
    'education_Bach_Degree_Pct',
    'education_Grad_Degree_Pct',
    'occupation_White_Collar_Pct',
    'occupation_Blue_Collar_Pct',
    'population_16P_Unemployed_Male_Pct',
    'population_16P_Unemployed_Female_Pct',
    'population_Chg_Pct_5_Yr_Projection',
    'population_Chg_Pct_2010',
    'population_White_Pct',
    'population_Black_Pct',
    'population_Hispanic_Pct',
    'population_Aged_25_34_Pct',
    'population_Aged_35_44_Pct',
    'population_Aged_65_74_Pct',

    'area_Square_Mile',
    'latitude',
    'longitude',
    'households_Family_W_Children_Pct',
    'population',
    'population_2020',
    'population_Male_Pct',
    'population_Female_Pct',
    'population_Aged_0_5_Pct',
    'population_Aged_6_11_Pct',
    'population_Aged_12_17_Pct',
    'population_Aged_18_24_Pct',
    'population_Aged_45_54_Pct',
    'population_Aged_55_64_Pct',
    'population_Aged_75_84_Pct',
    'population_Aged_85P_Pct',
    'costIndex_Annual_Expenditures',
    'costIndex_Housing',
    'costIndex_Healthcare',
    'costIndex_Education',
    'householder_Median_Age',
    'housing_Occupied_Structure_2_Units_Pct',
    'housing_Occupied_Structure_3_4_Units_Pct',
    'housing_Occupied_Structure_5_9_Units_Pct',
    'housing_Occupied_Structure_10_19_Units_Pct',
    'housing_Occupied_Structure_20_49_Units_Pct',
    'housing_Occupied_Structure_50_Or_More_Units_Pct',
    'housing_Built_2010_Or_Later_Pct',
    'housing_Built_2000_2009_Pct',
    'housing_Built_1990_1999_Pct',
    'housing_Built_1980_1989_Pct',
    'housing_Built_1970_1979_Pct',
    'housing_Built_1960_1969_Pct',
    'housing_Built_1950_1959_Pct',
    'housing_Built_1940_1949_Pct',
    'housing_Built_1939_Or_Earlier_Pct',
    'travel_Time_To_Work_0_14_Mi_Pct',
    'travel_Time_To_Work_15_29_Mi_Pct',
    'transportation_Walk_Pct',
    'transportation_Public_Pct',

    # school\district
    'schoolCnt',
    'schoolDistrictName',
    'gradeSpanLow',
    'gradeSpanHigh',
    'teacherCntFte',
    'studentCnt',
    'urbanCentricLocaleType',
    'totalPerPupilExpenditureAmt',
    'perPupilExpInstrPct',
    'perPupilExpSuptSvcsPct',
    'perPupilExpAdminPct',
    'studentYearlyChangePct',
    'popAge5_17BelowPovertyLevelPct',
    'schoolDistrictRating',
    'category',
    'health_cnt',
    'stores_cnt',
    'park_cnt',
    'hospitality_cnt',
    'education_cnt'
]

In [81]:
all_neighborhoods = pd.read_csv('all_neighborhoods.csv')

list_of_dicts = all_neighborhoods.to_dict('records')
# list_of_dicts

In [77]:
geo_ids = all_neighborhoods.loc[:, 'geo_id']
latitudes = all_neighborhoods.loc[:, 'latitude']
longitudes = all_neighborhoods.loc[:, 'longitude']

In [46]:
def get_community_data(geo_id):
  url = 'https://api.gateway.attomdata.com/v4/neighborhood/community'
  params = {'geoIdv4': geo_id}

  request = requests.get(url, headers=headers, params=params)
  data = request.json()

  return data['community']

In [47]:
def extract_features(community_data):
  geography = community_data['geography']
  demographics = community_data['demographics']
  crime = community_data['crime']
  climate = community_data['climate']
  air_quality = community_data['airQuality']

  result = {}
  for feature in features:
    if feature in geography:
      result[feature] = geography.get(feature)
    elif feature in demographics:
      result[feature] = demographics.get(feature)
    elif feature in crime:
      result[feature] = crime.get(feature)
    elif feature in climate:
      result[feature] = climate.get(feature)
    elif feature in air_quality:
      result[feature] = air_quality.get(feature)
    else:
      result[feature] = None
  return result

In [48]:
def get_all_data(list_of_dicts):

  all_data = []

  for data in list_of_dicts:
    iter_data = data.copy()
    community_data = get_community_data(iter_data['geo_id'])
    iter_data.update(extract_features(community_data))
    all_data.append(iter_data)
  return all_data

In [49]:
def get_geography_names(geography_name):
  all_names = geography_name.split(',')
  if len(all_names) == 3:
    return [None, all_names[0], all_names[1], all_names[2]]
  return all_names

Старая (без Thread Pool Executor):

In [74]:
# df = pd.DataFrame(get_all_data(list_of_dicts[:5]))
# splited = df['geographyName'].str.split(', ', expand=True)
# df[['neighborhood', 'city', 'county', 'state']] = df['geographyName'].apply(lambda x: pd.Series(get_geography_names(x)))
# df

Новая (c Thread Pool Executor):

In [71]:
def extract_data(row):
    community_data = get_community_data(row['geo_id'])
    features_data = extract_features(community_data)
    result = row.to_dict()
    result.update(features_data)
    return result

In [72]:
rows = [row for _, row in all_neighborhoods.iterrows()]

In [73]:
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

with ThreadPoolExecutor(max_workers=10) as executor:
    final_data = list(tqdm(executor.map(extract_data, rows[:100]), total=len(rows[:100])))

df = pd.DataFrame(final_data)

splited = df['geographyName'].str.split(', ', expand=True)
df[['neighborhood', 'city', 'county', 'state']] = df['geographyName'].apply(lambda x: pd.Series(get_geography_names(x)))

100%|██████████| 100/100 [00:09<00:00, 10.53it/s]


Пока нет признаков начиная с schoolCnt по education_cnt, потому что они лежат в school\district

In [69]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 95 columns):
 #   Column                                           Non-Null Count  Dtype  
---  ------                                           --------------  -----  
 0   geo_id                                           100 non-null    object 
 1   latitude                                         100 non-null    float64
 2   longitude                                        100 non-null    float64
 3   area                                             100 non-null    float64
 4   geographyName                                    100 non-null    object 
 5   median_Household_Income                          100 non-null    int64  
 6   household_Income_Per_Capita                      100 non-null    int64  
 7   housing_Median_Rent                              100 non-null    int64  
 8   housing_Owner_Households_Median_Value            100 non-null    int64  
 9   housing_Units_Renter_Occupied_Pct

In [75]:
df

,geo_id,latitude,longitude,area,geographyName,median_Household_Income,household_Income_Per_Capita,housing_Median_Rent,housing_Owner_Households_Median_Value,housing_Units_Renter_Occupied_Pct,...,category,health_cnt,stores_cnt,park_cnt,hospitality_cnt,education_cnt,neighborhood,city,county,state
0,c4985358751ed9e97f106aa9dc0aa640,39.753611,-105.112466,1.549311,"Acipco / Finley, Birmingham, Jefferson County, AL",33267,16911,475,60879,55.47,...,None,None,None,None,None,None,Acipco / Finley,Birmingham,Jefferson County,AL
1,7b8a4ad71938b16e13d0bc8e19fee22a,30.727964,-88.058559,0.828192,"Africatown, Mobile, Mobile County, AL",46193,11511,363,64402,50.70,...,None,None,None,None,None,None,Africatown,Mobile,Mobile County,AL
2,5df42aba382587eeadad52a615e8cef7,28.040854,-97.028924,1.587457,"Airmont, Mobile, Mobile County, AL",47083,24760,532,162720,53.82,...,None,None,None,None,None,None,Airmont,Mobile,Mobile County,AL
3,b060e1f941f76c9b6e6887a7db8ee21b,41.984866,-87.671462,0.270328,"Airport Highlands, Birmingham, Jefferson Count...",56903,21178,746,117295,59.93,...,None,None,None,None,None,None,Airport Highlands,Birmingham,Jefferson County,AL
4,7db649e182afe8c6c6595af9348909d7,29.290475,-81.084510,1.568713,"Alderbrook, Mobile County, AL",95479,33127,1136,291547,17.91,...,None,None,None,None,None,None,None,Alderbrook,Mobile County,AL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,b505b73ca323744eb40d1bcc173161af,39.335510,-104.862626,0.858895,"East Lake, Birmingham, Jefferson County, AL",30896,17162,582,88721,55.18,...,None,None,None,None,None,None,East Lake,Birmingham,Jefferson County,AL
96,250b0b3df33d1ee1ad7a918b4d011823,29.500591,-98.346924,0.416079,"East Mastin Lake, Huntsville, Madison County, AL",33540,17975,661,91073,53.72,...,None,None,None,None,None,None,East Mastin Lake,Huntsville,Madison County,AL
97,8f47b8f892286a08a09c788894927220,39.742930,-105.065290,0.428226,"East Thomas, Birmingham, Jefferson County, AL",44837,25881,495,78204,26.48,...,None,None,None,None,None,None,East Thomas,Birmingham,Jefferson County,AL
98,27f5a303a1c67af7c066899454f89998,39.105017,-104.803921,1.112973,"Eastwood, Birmingham, Jefferson County, AL",58791,34208,812,193975,63.17,...,None,None,None,None,None,None,Eastwood,Birmingham,Jefferson County,AL
